# Phase 2: CoT Implementation - Executable Deep Dive
## Building Production-Ready CoT Systems

In this phase, you'll:
1. Set up benchmark datasets
2. Implement a full CoT evaluator
3. Test on real benchmarks
4. Optimize hyperparameters
5. Create production-ready code

**Time Estimate:** 4-6 hours with full benchmarking

## Section 1: Setup and Imports

In [ ]:
import os
import json
import re
import time
from typing import Dict, List, Tuple
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 7)

print("✅ Imports successful")

## Section 2: Load Local Model (Mistral-7B)

In [ ]:
# Load local Mistral-7B model (same as Phase 1)
from transformers import pipeline
import torch

print("Loading Mistral-7B model (local inference, no API needed)...")
print("(First time takes ~2-3 minutes, ~4GB download)")

try:
    device = 0 if torch.cuda.is_available() else -1
    device_str = "GPU" if device == 0 else "CPU"
    print(f"Using: {device_str}")
    
    generator = pipeline(
        "text-generation",
        model="mistralai/Mistral-7B-Instruct-v0.1",
        torch_dtype=torch.float16 if device == 0 else torch.float32,
        device_map="auto" if device == 0 else None,
        device=device
    )
    
    print("\n✅ Mistral-7B model loaded successfully!")
    
    def call_model(prompt: str, temperature: float = 0.0, max_tokens: int = 500) -> str:
        """Generate text using local Mistral-7B model"""
        try:
            # Handle temperature=0.0 for greedy decoding
            use_sampling = temperature > 0.0
            actual_temp = max(temperature, 0.1) if not use_sampling else temperature
            
            response = generator(
                prompt,
                max_new_tokens=max_tokens,
                temperature=actual_temp,
                num_return_sequences=1,
                do_sample=use_sampling
            )
            return response[0]["generated_text"]
        except Exception as e:
            raise e
    
    print("✅ Local model call_model() function ready")
    
except Exception as e:
    print(f"❌ Error loading Mistral-7B: {e}")
    print("\nFallback: Using smaller GPT-2 model...")
    
    generator = pipeline("text-generation", model="gpt2", device=-1)
    
    def call_model(prompt: str, temperature: float = 0.0, max_tokens: int = 500) -> str:
        """Generate text using fallback GPT-2 model"""
        try:
            use_sampling = temperature > 0.0
            actual_temp = max(temperature, 0.1) if not use_sampling else temperature
            
            response = generator(
                prompt,
                max_new_tokens=max_tokens,
                temperature=actual_temp,
                num_return_sequences=1,
                do_sample=use_sampling
            )
            return response[0]["generated_text"]
        except Exception as e:
            raise e

## Section 3: Create CoT Evaluator Class

In [ ]:
class CoTEvaluator:
    """Production-ready CoT evaluator"""
    
    def __init__(self, model_fn, name: str = "default"):
        self.model = model_fn
        self.name = name
        self.results = []
    
    def extract_answer(self, response: str) -> str:
        """Extract numerical answer from response"""
        # Look for patterns like "The answer is X" or just numbers
        patterns = [
            r'(?:the )?answer is[:\s]+([\d]+)',  # "The answer is X"
            r'(?:so|therefore).*?([\d]+)\s*(?:apples|items|books|dollars|people)?$',  # "So X apples"
            r'([\d]+)\s*(?:apples|items|books|dollars|people|brothers|sisters|children|cars)?[.\s]*$'  # Ending with "X items"
        ]
        
        response_lower = response.lower()
        
        for pattern in patterns:
            match = re.search(pattern, response_lower)
            if match:
                return match.group(1)
        
        # Fallback: return last number
        numbers = re.findall(r'\d+', response)
        return numbers[-1] if numbers else ""
    
    def evaluate_single(self, question: str, expected_answer: str, use_cot: bool = True, 
                       temperature: float = 0.0, max_tokens: int = 300) -> Dict:
        """Evaluate a single question"""
        
        if use_cot:
            prompt = f"Let me work through this step by step.\n\nQ: {question}\nA:"
        else:
            prompt = f"Q: {question}\nA:"
        
        try:
            response = self.model(prompt, temperature=temperature, max_tokens=max_tokens)
            predicted = self.extract_answer(response)
            is_correct = predicted == expected_answer
            
            return {
                'question': question,
                'expected': expected_answer,
                'predicted': predicted,
                'response': response,
                'correct': is_correct,
                'method': 'cot' if use_cot else 'standard',
                'temperature': temperature,
                'timestamp': datetime.now().isoformat()
            }
        except Exception as e:
            return {
                'question': question,
                'expected': expected_answer,
                'error': str(e),
                'method': 'cot' if use_cot else 'standard'
            }
    
    def evaluate_batch(self, questions: List[str], answers: List[str], use_cot: bool = True,
                      batch_name: str = "batch") -> Dict:
        """Evaluate a batch of questions"""
        results = []
        
        print(f"\n" + "="*80)
        print(f"Evaluating {len(questions)} questions with {'CoT' if use_cot else 'Standard'} prompting")
        print(f"Batch: {batch_name}")
        print("="*80)
        
        for i, (q, a) in enumerate(tqdm(zip(questions, answers), total=len(questions))):
            result = self.evaluate_single(q, a, use_cot=use_cot)
            results.append(result)
            time.sleep(0.1)  # Rate limiting
        
        # Calculate metrics
        correct = sum(1 for r in results if r.get('correct', False))
        total = len(results)
        accuracy = correct / total if total > 0 else 0
        
        summary = {
            'batch': batch_name,
            'method': 'cot' if use_cot else 'standard',
            'total': total,
            'correct': correct,
            'accuracy': accuracy,
            'results': results
        }
        
        print(f"\n✅ Accuracy: {accuracy*100:.1f}% ({correct}/{total})")
        
        return summary

# Create evaluator
evaluator = CoTEvaluator(call_model, name="gpt-3.5-turbo")
print("✅ CoTEvaluator created")

## Section 4: Load Benchmark Dataset

In [ ]:
# Small sample benchmark (for quick testing)
# In production, use: load_dataset('gsm8k', 'main')

benchmark_questions = [
    # GSM8K-style problems
    ("Natalia sold clips to 48 of her friends in April, and then she sold clips to 42 of her friends in May. How many friends did Natalia sell clips to?", "90"),
    ("Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?", "10"),
    ("Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. How much more money does Betty need to save?", "50"),
    ("Julie is reading a book that has 120 pages. She has already read 30 pages. If she reads 5 pages a day, how many days will it take her to finish the book?", "18"),
    ("John buys 3 apples. Mary buys 5 apples. How many apples do they have together?", "8"),
    ("A store has 200 apples. They sell 30 apples on Monday, 25 apples on Tuesday, and 40 apples on Wednesday. How many apples are left?", "105"),
    ("Tom has twice as many apples as Jerry. Jerry has 15 apples. How many apples does Tom have?", "30"),
    ("Sarah has 10 dollars. She buys a book for 3 dollars and a pen for 2 dollars. How much money does she have left?", "5"),
    ("A car travels 60 miles in 1 hour. How far will it travel in 3 hours?", "180"),
    ("A pizza is divided into 8 slices. If 3 people each eat 2 slices, how many slices are left?", "2"),
]

questions, answers = zip(*benchmark_questions)
questions = list(questions)
answers = list(answers)

print(f"✅ Loaded {len(questions)} benchmark questions")

## Section 5: Run Evaluations

In [ ]:
# Evaluate with standard prompting
standard_results = evaluator.evaluate_batch(
    questions[:1],  # Test on first 5 (changing 1 due to long inference latency on local machine) for quick evaluation
    answers[:1],
    use_cot=False,
    batch_name="Standard Prompting"
)

In [ ]:
# Evaluate with CoT prompting
cot_results = evaluator.evaluate_batch(
    questions[:1],  # Same questions
    answers[:1],
    use_cot=True,
    batch_name="Chain-of-Thought Prompting"
)

## Section 6: Compare Results

In [ ]:
# Create comparison
print("\n" + "="*80)
print("RESULTS COMPARISON")
print("="*80)

comparison_data = {
    'Method': ['Standard', 'CoT'],
    'Correct': [standard_results['correct'], cot_results['correct']],
    'Total': [standard_results['total'], cot_results['total']],
    'Accuracy %': [standard_results['accuracy']*100, cot_results['accuracy']*100]
}

df_comparison = pd.DataFrame(comparison_data)
print("\n", df_comparison.to_string(index=False))

improvement = cot_results['accuracy']*100 - standard_results['accuracy']*100
print(f"\n📈 Improvement: {improvement:+.1f}%")

## Section 7: Analyze Examples

In [ ]:
# Show example responses
print("\n" + "="*80)
print("EXAMPLE RESPONSES")
print("="*80)

# Compare first question
std_result = standard_results['results'][0]
cot_result = cot_results['results'][0]

print(f"\n📝 Question: {std_result['question']}")
print(f"✅ Expected Answer: {std_result['expected']}")

print(f"\n" + "-"*80)
print("STANDARD PROMPTING")
print("-"*80)
print(f"Predicted: {std_result['predicted']}")
print(f"Correct: {'✓' if std_result['correct'] else '✗'}")
print(f"\nResponse: {std_result['response'][:200]}...")

print(f"\n" + "-"*80)
print("CHAIN-OF-THOUGHT PROMPTING")
print("-"*80)
print(f"Predicted: {cot_result['predicted']}")
print(f"Correct: {'✓' if cot_result['correct'] else '✗'}")
print(f"\nResponse: {cot_result['response'][:200]}...")

## Section 8: Hyperparameter Optimization

In [ ]:
# Test different temperatures
print("\n" + "="*80)
print("HYPERPARAMETER TUNING: Temperature")
print("="*80)

#temperatures = [0.0, 0.3, 0.7]
temperatures = [0.0]
temp_results = []

for temp in temperatures:
    results = evaluator.evaluate_batch(
        questions[:2],  # Smaller set for quicker testing
        answers[:2],
        use_cot=True,
        batch_name=f"CoT with Temperature={temp}"
    )
    temp_results.append({
        'temperature': temp,
        'accuracy': results['accuracy'] * 100
    })

df_temps = pd.DataFrame(temp_results)
print("\n", df_temps.to_string(index=False))

# Plot
plt.figure(figsize=(10, 5))
plt.plot(df_temps['temperature'], df_temps['accuracy'], marker='o', linewidth=2, markersize=8, color='#4ECDC4')
plt.xlabel('Temperature', fontsize=12, fontweight='bold')
plt.ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
plt.title('Effect of Temperature on CoT Accuracy', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.ylim(0, 100)
plt.tight_layout()
plt.savefig('temperature_tuning.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Temperature tuning plot saved")

## Section 9: Save Results

In [ ]:
# Save comprehensive results
results_file = 'phase2_results.json'

save_data = {
    'timestamp': datetime.now().isoformat(),
    'standard_accuracy': standard_results['accuracy'],
    'cot_accuracy': cot_results['accuracy'],
    'improvement': improvement / 100,
    'standard_results': standard_results['results'],
    'cot_results': cot_results['results'],
    'temperature_tuning': temp_results
}

with open(results_file, 'w') as f:
    json.dump(save_data, f, indent=2)

print(f"✅ Results saved to {results_file}")

# Also save as CSV for easy analysis
df_all = pd.DataFrame(standard_results['results'] + cot_results['results'])
df_all.to_csv('phase2_results.csv', index=False)
print("✅ Results saved to phase2_results.csv")

## Section 10: Production Code - Reusable CoT Solver

In [ ]:
class ProductionCoTSolver:
    """Production-ready CoT solver with caching and error handling"""
    
    def __init__(self, model_fn, cache_enabled: bool = True):
        self.model = model_fn
        self.cache = {} if cache_enabled else None
        self.cache_hits = 0
        self.cache_misses = 0
    
    def _get_cot_examples(self, domain: str = "math") -> str:
        """Get CoT examples based on domain"""
        examples = {
            'math': '''Let me work through math problems step by step.

Example 1:
Q: Sarah has 5 oranges and buys 3 more. How many does she have?
A: Sarah starts with 5 oranges. She buys 3 more. So 5 + 3 = 8 oranges.

Example 2:
Q: A box contains 20 items. 7 are removed. How many remain?
A: Starting with 20 items. 7 are removed. So 20 - 7 = 13 items remain.''',
            
            'logic': '''Let me reason through logic problems carefully.

Example 1:
Q: All dogs are animals. Max is a dog. Is Max an animal?
A: All dogs are animals (given). Max is a dog (given). Therefore, Max must be an animal.'''
        }
        return examples.get(domain, examples['math'])
    
    def solve(self, problem: str, domain: str = "math", use_cache: bool = True) -> Dict:
        """Solve a problem with CoT"""
        
        # Check cache
        if use_cache and self.cache is not None and problem in self.cache:
            self.cache_hits += 1
            return self.cache[problem]
        
        self.cache_misses += 1
        
        # Build prompt
        examples = self._get_cot_examples(domain)
        prompt = f"{examples}\n\nQ: {problem}\nA:"
        
        try:
            response = self.model(prompt, temperature=0.0, max_tokens=300)
            
            result = {
                'problem': problem,
                'response': response,
                'status': 'success',
                'timestamp': datetime.now().isoformat()
            }
            
            # Cache the result
            if use_cache and self.cache is not None:
                self.cache[problem] = result
            
            return result
        
        except Exception as e:
            return {
                'problem': problem,
                'error': str(e),
                'status': 'error',
                'timestamp': datetime.now().isoformat()
            }
    
    def get_cache_stats(self) -> Dict:
        """Get cache statistics"""
        total = self.cache_hits + self.cache_misses
        return {
            'cache_hits': self.cache_hits,
            'cache_misses': self.cache_misses,
            'total_requests': total,
            'hit_rate': self.cache_hits / total if total > 0 else 0
        }

# Create solver instance
solver = ProductionCoTSolver(call_model)
print("✅ ProductionCoTSolver created")

In [ ]:
# Test the production solver
test_problems = [
    "If John has 10 apples and gives 3 to Mary, how many does he have left?",
    "A store has 50 books. They sell 12 on Monday and 8 on Tuesday. How many books remain?"
]

print("\n" + "="*80)
print("TESTING PRODUCTION SOLVER")
print("="*80)

for problem in test_problems:
    result = solver.solve(problem, domain="math")
    print(f"\n📝 Problem: {problem}")
    print(f"✅ Response: {result['response'][:150]}...")

print("\n" + "-"*80)
print(f"Cache Stats: {solver.get_cache_stats()}")

## Section 11: Summary and Next Steps

In [ ]:
summary = f"""
╔════════════════════════════════════════════════════════════════════════════════╗
║                      PHASE 2 SUMMARY                                           ║
╚════════════════════════════════════════════════════════════════════════════════╝

✅ WHAT YOU'VE BUILT

1. CoTEvaluator Class
   - Evaluate standard vs CoT prompting
   - Extract answers from responses
   - Calculate accuracy metrics
   - Generate detailed reports

2. Benchmark Evaluation
   - Tested on 5 math problems
   - Compared two prompting methods
   - Measured performance improvements
   - Analyzed response quality

3. Hyperparameter Tuning
   - Tested different temperatures
   - Found optimal settings
   - Visualized results
   - Documented findings

4. Production-Ready Code
   - Created reusable ProductionCoTSolver
   - Implemented caching for efficiency
   - Added error handling
   - Cache statistics tracking

📊 RESULTS

- Standard Accuracy: {standard_results['accuracy']*100:.1f}%
- CoT Accuracy: {cot_results['accuracy']*100:.1f}%
- Improvement: {improvement:+.1f}%

🔧 PRODUCTION FEATURES

✓ Caching to avoid duplicate API calls
✓ Error handling and recovery
✓ Domain-specific CoT examples
✓ Configurable parameters
✓ Performance metrics tracking

📝 FILES CREATED

- phase2_results.json - Complete results
- phase2_results.csv - Easy analysis format
- temperature_tuning.png - Visualization
- ProductionCoTSolver class - Reusable code

🎯 READY FOR PHASE 3

In Phase 3, you'll learn:
- Self-Consistency Sampling (multiple reasoning paths)
- Tree-of-Thoughts (exploring reasoning trees)
- Multi-model ensembles
- Advanced optimization techniques

Let's go to Phase 3!
"""

print(summary)